# Tradepad
Option valuation on one underlying, OVME-style: define the deal, see the market inputs, price it with Black-Scholes
and the calibrated equity models (same functions as the web app, `tqstudio/views/tradepad.py`).

## Deal

In [ ]:
from datetime import date

TICKER = "TICKER1"
PRODUCT = "Vanilla"            # Vanilla | Digital | Barrier
OPTION_TYPE = "Put"            # Call | Put
STRIKE_PCT = 80.0              # strike / spot, %
EXPIRY = date(2027, 11, 5)
QUANTITY = 1.0
# Digital
PAYOFF, CASH_AMOUNT = "Cash", 100.0          # Cash | Asset
# Barrier (continuous monitoring)
BARRIER_TYPE, BARRIER_PCT = "DownIn", 70.0   # DownIn | DownOut | UpIn | UpOut

MODELS = ["Black-Scholes", "Local vol", "Heston", "Displaced diffusion", "Heston SLV"]
N_PATHS = 50_000

In [ ]:
import os
import sys
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "marketdata").is_dir())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import tensorquant as tq
from tqstudio.views.tradepad import ModelCache, price_trade

mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")       # latest snapshot
tq.Settings.evaluation_date = mkt.evaluation_date
cache = ModelCache()                                               # calibrated models, reused across pricings
trade = dict(ticker=TICKER, product=PRODUCT, option_type=OPTION_TYPE, strike_pct=STRIKE_PCT,
             expiry=EXPIRY.isoformat(), quantity=QUANTITY, payoff=PAYOFF, cash_amount=CASH_AMOUNT,
             barrier_type=BARRIER_TYPE, barrier_pct=BARRIER_PCT)

## Valuation

In [ ]:
result = price_trade(mkt, cache, trade, MODELS, n_paths=N_PATHS)
t = result["trade"]
eq = mkt.get_equity(TICKER)


underlying = {"ticker": TICKER, "currency": eq.currency.name, "snapshot": mkt.evaluation_date,
                                  "spot": round(t["spot"], 2)}
deal = {"product": PRODUCT, "type": OPTION_TYPE, "strike": f"{t['strike']:.2f} ({STRIKE_PCT:g}%)",
        "expiry": f"{t['expiry']} ({t['T']:.2f}y)", "quantity": QUANTITY}
if PRODUCT == "Digital":
    deal["payoff"] = f"{PAYOFF}" + (f" {CASH_AMOUNT:g}" if PAYOFF == "Cash" else "")
if PRODUCT == "Barrier":
    deal["barrier"] = f"{BARRIER_TYPE} {t['barrier']:.2f} ({BARRIER_PCT:g}%), continuous"
market = {"forward": round(t["forward"], 2), "rate (zero, %)": round(100 * t["rate"], 4),
                          "discount factor": round(t["discount_factor"], 5), "repo (%)": round(100 * t["repo"], 4),
                          "PV dividends": round(t["pv_dividends"], 4),
                          "implied div yield (%)": round(100 * t["implied_div_yield"], 4),
                          "vol at strike (%)": round(100 * t["vol_at_strike"], 2),
                          "vol ATM forward (%)": round(100 * t["vol_atm_forward"], 2)}
model = {"Monte Carlo": f"{t['n_paths']:,} paths, {t['n_steps']} steps",
                        "greeks": "AAD through the calibration", "Heston params": "held (SLV)"}
panels = {"Underlying": underlying, "Deal": deal, "Market": market, "Model": model}
pd.concat({k: pd.Series(v, dtype=object) for k, v in panels.items()}).to_frame("value")

In [ ]:
rows = []
for r in result["rows"]:
    if "error" in r:
        rows.append({"model": r["model"], "error": r["error"]})
        continue
    rows.append({"model": r["model"], "price": r["price"], "premium (% spot)": 100 * r["unit_price"] / t["spot"],
                 "s.e.": r["standard_error"], "delta": r["delta"], "delta (%)": 100 * r["delta"] / QUANTITY,
                 "vega (1 vol pt)": r["vega"], "rho (1 bp)": r["rho"],
                 "calibration [s]": r["calibration_s"], "pricing [s]": r["pricing_s"]})
valuation = pd.DataFrame(rows).set_index("model")
ok = valuation["price"].notna()
valuation.insert(1, "vs " + valuation.index[ok][0], valuation["price"] - valuation["price"][ok].iloc[0])
valuation.round(4)

- **price**: per unit times the quantity; **premium**: unit price as % of spot; **s.e.**: Monte Carlo standard error.
- **delta**: dV/dS (shares); **vega**: sum of the vega per surface node, per vol point; **rho**: sum over the
  discount-curve pillars, per bp. All through the model calibrations (the hedge ratios for the market quotes).
- First pricing of a model calibrates it (Heston SLV reuses the Heston and local vol ones); later pricings reuse it.

## Vega per surface node

In [ ]:
grid = result["vega_grid"]
ok_rows = [r for r in result["rows"] if "error" not in r]
maps = {r["model"]: np.asarray(r["vega_nodes"]) for r in ok_rows}
scale = max(np.abs(v).max() for v in maps.values())
fig, axes = plt.subplots(1, len(maps), figsize=(3.3 * len(maps), 3.4), sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (name, v) in zip(axes, maps.items()):
    im = ax.imshow(v, cmap="RdBu_r", vmin=-scale, vmax=scale, aspect="auto")
    for (i, j), x in np.ndenumerate(v):
        if abs(x) >= 0.01 * scale:
            ax.text(j, i, f"{x:.2f}", ha="center", va="center", fontsize=6.5)
    ax.set_title(name, fontsize=10)
    ax.set_xticks(range(len(grid["moneyness"])), [f"{100 * m:.0f}" for m in grid["moneyness"]], fontsize=7)
    ax.set_xlabel("K / S0 %", fontsize=8)
axes[0].set_yticks(range(len(grid["expiries"])), [f"{x:.2f}" for x in grid["expiries"]], fontsize=7)
axes[0].set_ylabel("T (years)", fontsize=8)
fig.colorbar(im, ax=axes, shrink=0.85, label="vega per vol point");

## Reprice another deal

In [ ]:
# the calibrated models are cached: change the deal and price again in a few seconds
trade2 = {**trade, "product": "Digital", "strike_pct": 90.0}
result2 = price_trade(mkt, cache, trade2, MODELS, n_paths=N_PATHS)
pd.DataFrame([{"model": r["model"], "price": r.get("price"), "delta": r.get("delta"), "vega": r.get("vega"),
               "calibration [s]": r["calibration_s"], "pricing [s]": r.get("pricing_s")}
              for r in result2["rows"]]).set_index("model").round(4)